# 01 — Numerical validation and version baseline

This notebook validates the maintained production implementation against the
high-accuracy reference implementation using the physical LISA orbit file.

It also records or compares a compact numerical baseline. All baseline and
report files are saved directly beside the notebook with a `_local_wdm_`
prefix.

Run it once with `BASELINE_ACTION = "record"` after a trusted commit. On later
commits, switch to `"compare"`.

In [1]:

import jax
import jax.numpy as jnp

# The reference backend genuinely uses complex128/float64.
jax.config.update("jax_enable_x64", True)

import json
import platform
import subprocess
import time
from pathlib import Path

import jax
import numpy as np

from lisaorbits import InterpolatedOrbits
from lisaorbits.file import read_orbit_file

import WDM_LISAresponse


NOTEBOOK_DIR = Path.cwd().resolve()
ORBIT_FILE = NOTEBOOK_DIR / "esa-trailing-orbits.h5"

if not ORBIT_FILE.exists():
    raise FileNotFoundError(
        "Place esa-trailing-orbits.h5 in the same directory as these notebooks. "
        f"Current notebook directory: {NOTEBOOK_DIR}"
    )


def load_lisa_orbits(path: Path):
    """Load the physical LISA orbit file used by the response calculations."""
    orbit_data = read_orbit_file(str(path))
    orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
    orbits = InterpolatedOrbits(
        orbit_times,
        orbit_data.positions,
        orbit_data.velocities,
    )
    return orbit_data, orbits


def git_state(package_file: str) -> dict[str, object]:
    """Return the current Git commit and whether the checkout is dirty."""
    completed = subprocess.run(
        [
            "git",
            "-C",
            str(Path(package_file).resolve().parent),
            "rev-parse",
            "--show-toplevel",
        ],
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        return {"root": None, "commit": None, "dirty": None}

    root = Path(completed.stdout.strip()).resolve()
    commit = subprocess.run(
        ["git", "-C", str(root), "rev-parse", "HEAD"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    dirty = bool(
        subprocess.run(
            ["git", "-C", str(root), "status", "--porcelain"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    )
    return {"root": str(root), "commit": commit, "dirty": dirty}


def json_ready(value):
    """Convert NumPy and JAX objects into JSON-compatible values."""
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if hasattr(value, "item") and getattr(value, "ndim", None) == 0:
        return value.item()
    if hasattr(value, "tolist"):
        return value.tolist()
    return value


def save_json(filename: str, payload) -> Path:
    """Save a JSON result beside the notebook."""
    path = NOTEBOOK_DIR / filename
    path.write_text(
        json.dumps(json_ready(payload), indent=2),
        encoding="utf-8",
    )
    print("Saved:", path)
    return path


orbit_data, orbits = load_lisa_orbits(ORBIT_FILE)
YEAR_SI = 365.25 * 24.0 * 3600.0
t0 = float(orbit_data.t0 + YEAR_SI)

print("Notebook directory:", NOTEBOOK_DIR)
print("Orbit file:", ORBIT_FILE)
print("Python:", platform.python_version())
print("JAX:", jax.__version__, "| backend:", jax.default_backend())
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("Source state:", git_state(WDM_LISAresponse.__file__))


from WDM_LISAresponse.code.lisa_response.TDI_with_one_arm_plan import (
    gen2_michelson_wdm_from_waveform_precomputed,
    precompute_gen2_michelson_wdm_geometry,
    precompute_gen2_michelson_wdm_waveform,
)
from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)

def block_result(value):
    """Synchronise every JAX leaf before stopping a timer."""
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return value


def timed(function):
    started = time.perf_counter()
    value = function()
    block_result(value)
    return value, float(time.perf_counter() - started)


def timing_summary(values):
    values = np.asarray(values, dtype=float)
    q25, q75 = np.percentile(values, [25.0, 75.0])
    return {
        "median_s": float(np.median(values)),
        "q25_s": float(q25),
        "q75_s": float(q75),
        "iqr_s": float(q75 - q25),
        "minimum_s": float(np.min(values)),
        "maximum_s": float(np.max(values)),
        "all_s": [float(value) for value in values],
    }


def benchmark(function, *, warmups: int, repeats: int):
    for _ in range(int(warmups)):
        block_result(function())

    samples = []
    latest = None
    for _ in range(int(repeats)):
        latest, elapsed = timed(function)
        samples.append(elapsed)
    return latest, timing_summary(samples)


def compare_arrays(candidate, reference, *, m_start: int = 0):
    """Return max-absolute and relative-L2 differences."""
    candidate = jax.numpy.asarray(candidate)[..., m_start:]
    reference = jax.numpy.asarray(reference)[..., m_start:]
    difference = candidate - reference

    difference_power = jax.numpy.sum(jax.numpy.abs(difference) ** 2)
    reference_power = jax.numpy.sum(jax.numpy.abs(reference) ** 2)
    max_abs = jax.numpy.max(jax.numpy.abs(difference))
    rel_l2 = jax.numpy.where(
        reference_power > 0,
        jax.numpy.sqrt(difference_power / reference_power),
        jax.numpy.sqrt(difference_power),
    )
    block_result((max_abs, rel_l2))
    return {"max_abs": float(max_abs), "rel_l2": float(rel_l2)}


def compare_link_maps(candidate, reference, *, m_start: int = 0):
    if set(candidate) != set(reference):
        raise ValueError("Directed-link dictionaries contain different keys.")

    metrics = {
        str(link): compare_arrays(
            candidate[link],
            reference[link],
            m_start=m_start,
        )
        for link in reference
    }
    return {
        "per_link": metrics,
        "max_abs": max(item["max_abs"] for item in metrics.values()),
        "rel_l2": max(item["rel_l2"] for item in metrics.values()),
    }


Notebook directory: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples
Orbit file: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples/esa-trailing-orbits.h5
Python: 3.13.11
JAX: 0.7.0 | backend: cpu
WDM_LISAresponse: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/src/WDM_LISAresponse/__init__.py
Source state: {'root': '/Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse', 'commit': '155979a0a509bc82690c867631655d2434783661', 'dirty': True}


## 1. Baseline mode and compact fixture

- `record`: create a new trusted baseline. Existing files are protected
  unless `ALLOW_BASELINE_OVERWRITE=True`.
- `compare`: load the existing baseline and compare the current revision.
- `none`: run only the internal production/reference checks.

Keep the same `BASELINE_NAME` for revisions that should be compared.


In [2]:
BASELINE_ACTION = "record"  # "record", "compare", or "none"
BASELINE_NAME = "clean_one_arm_v1"
ALLOW_BASELINE_OVERWRITE = False

N = 16_384
dt = 10.0
Nf = 32
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25
CALC_M0 = False
M_START = 1

theta = 0.8
phi = 0.5

PRODUCTION_SHIFT = {
    "assembly_backend": "production",
    "assembly_precision": "complex64",
    "row_chunk_size": 256,
    "lag_block_size": 4,
    "batch_chunk": 32,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 16,
    "tl_tp_interp_kind": "linear",
}
REFERENCE_SHIFT = {
    "assembly_backend": "reference",
    "assembly_precision": "complex128",
    "row_chunk_size": 128,
    "lag_block_size": 1,
    "batch_chunk": 1,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 16,
    "tl_tp_interp_kind": "linear",
}

times = t0 + np.arange(N, dtype=float) * dt
times_pad, h_plus, h_cross, padding = generate_windowed_padded_waveform(
    waveform_model="linear_freq",
    t0=t0,
    dt=dt,
    N=N,
    model_kwargs={
        "A_plus": 1.0e-21,
        "A_cross": 0.8e-21,
        "f_start": 8.0e-4,
        "f_end": 3.0e-3,
        "phi_0": 0.2,
    },
    window_function="tukey",
    window_kwargs={"alpha": 0.02, "tukey_factor": 2.0},
    wdm_Nf=Nf,
    return_padding_info=True,
)

print("Padded samples:", times_pad.size)
print("WDM shape:", (padding["wdm_Nt"], Nf))


Padded samples: 17088
WDM shape: (534, 32)


In [3]:
def run_staged(
    shift_kwargs,
    *,
    execution_mode="materialized",
    links_per_group=3,
    max_workers=2,
):
    geometry = precompute_gen2_michelson_wdm_geometry(
        times_pad,
        theta,
        phi,
        orbits,
        Nf=Nf,
        L_trunc=L_TRUNC,
        q=q_wdm,
        d=d_wdm,
        A_frac=A_frac,
        calc_m0=CALC_M0,
        variable_shift_kwargs=shift_kwargs,
    )
    waveform_state = precompute_gen2_michelson_wdm_waveform(
        geometry,
        h_plus,
        h_cross,
        return_device_links=True,
        one_arm_execution_mode=execution_mode,
        one_arm_parallel_links_per_group=links_per_group,
        one_arm_parallel_max_workers=max_workers,
        parallel_wdm_transforms=False,
    )
    xyz = gen2_michelson_wdm_from_waveform_precomputed(
        waveform_state,
        tdi_assembly_mode="prepared_plan",
    )
    block_result((waveform_state["link_w"], xyz))
    return geometry, waveform_state, xyz


production_geometry, production_wave, production_xyz = run_staged(
    PRODUCTION_SHIFT
)
reference_geometry, reference_wave, reference_xyz = run_staged(
    REFERENCE_SHIFT
)

print("Production backend:",
      production_geometry["one_arm_plan"].shift_plan.resolved_assembly_backend)
print("Reference backend:",
      reference_geometry["one_arm_plan"].shift_plan.resolved_assembly_backend)


Production backend: production
Reference backend: reference


## 2. Production versus reference

The production path differs in two controlled ways:

- interpolated \(T_\ell,T'_\ell\);
- complex64/float32 arithmetic and analytic-parity assembly.

Therefore this comparison is approximate. It is evaluated only on
`m >= 1`.


In [4]:
production_reference_links = compare_link_maps(
    production_wave["link_w"],
    reference_wave["link_w"],
    m_start=M_START,
)
production_reference_xyz = {
    name: compare_arrays(candidate, reference, m_start=M_START)
    for name, candidate, reference in zip(
        ("X", "Y", "Z"),
        production_xyz,
        reference_xyz,
    )
}

print("One-arm production/reference:")
print(json.dumps(json_ready(production_reference_links), indent=2))
print("TDI production/reference:")
print(json.dumps(json_ready(production_reference_xyz), indent=2))

# These are regression guards, not final scientific tolerances.
assert production_reference_links["rel_l2"] < 5.0e-4
assert max(item["rel_l2"] for item in production_reference_xyz.values()) < 1.0e-3


One-arm production/reference:
{
  "per_link": {
    "(2, 1)": {
      "max_abs": 4.173284089738556e-18,
      "rel_l2": 1.5926456216539718e-07
    },
    "(3, 2)": {
      "max_abs": 1.1028342915850005e-19,
      "rel_l2": 1.6355213359672242e-07
    },
    "(1, 3)": {
      "max_abs": 5.2718649655518685e-18,
      "rel_l2": 1.525332419866391e-07
    },
    "(3, 1)": {
      "max_abs": 6.498572853854867e-18,
      "rel_l2": 1.6570365709735197e-07
    },
    "(2, 3)": {
      "max_abs": 1.1656531151201788e-19,
      "rel_l2": 1.7540656721177795e-07
    },
    "(1, 2)": {
      "max_abs": 3.593052279468775e-18,
      "rel_l2": 1.504639287196372e-07
    }
  },
  "max_abs": 6.498572853854867e-18,
  "rel_l2": 1.7540656721177795e-07
}
TDI production/reference:
{
  "X": {
    "max_abs": 1.0719026294730743e-16,
    "rel_l2": 7.316445886957545e-05
  },
  "Y": {
    "max_abs": 3.454852351070982e-16,
    "rel_l2": 9.06994059722738e-05
  },
  "Z": {
    "max_abs": 4.3580980428699784e-16,
    "rel_l

## 3. Maintained one-arm scheduling equivalence

`materialized_parallel_links` changes CPU scheduling only. It must produce
the same six link arrays as `materialized`.


In [5]:
parallel_wave = precompute_gen2_michelson_wdm_waveform(
    production_geometry,
    h_plus,
    h_cross,
    return_device_links=True,
    one_arm_execution_mode="materialized_parallel_links",
    one_arm_parallel_links_per_group=3,
    one_arm_parallel_max_workers=2,
    parallel_wdm_transforms=False,
)
block_result(parallel_wave["link_w"])

scheduling_comparison = compare_link_maps(
    parallel_wave["link_w"],
    production_wave["link_w"],
    m_start=M_START,
)
print(json.dumps(json_ready(scheduling_comparison), indent=2))
assert scheduling_comparison["rel_l2"] == 0.0


{
  "per_link": {
    "(2, 1)": {
      "max_abs": 0.0,
      "rel_l2": 0.0
    },
    "(3, 2)": {
      "max_abs": 0.0,
      "rel_l2": 0.0
    },
    "(1, 3)": {
      "max_abs": 0.0,
      "rel_l2": 0.0
    },
    "(3, 1)": {
      "max_abs": 0.0,
      "rel_l2": 0.0
    },
    "(2, 3)": {
      "max_abs": 0.0,
      "rel_l2": 0.0
    },
    "(1, 2)": {
      "max_abs": 0.0,
      "rel_l2": 0.0
    }
  },
  "max_abs": 0.0,
  "rel_l2": 0.0
}


## 4. Prepared TDI versus retained batch-map reference

The prepared plan is the production B stage. `batch_map` remains a
validation route that reconstructs the shift work at runtime.


In [6]:
batch_map_xyz = gen2_michelson_wdm_from_waveform_precomputed(
    production_wave,
    tdi_assembly_mode="batch_map",
)
block_result(batch_map_xyz)

prepared_batch_map = {
    name: compare_arrays(prepared, batch, m_start=M_START)
    for name, prepared, batch in zip(
        ("X", "Y", "Z"),
        production_xyz,
        batch_map_xyz,
    )
}
print(json.dumps(json_ready(prepared_batch_map), indent=2))
assert max(item["rel_l2"] for item in prepared_batch_map.values()) < 2.0e-5


{
  "X": {
    "max_abs": 1.3877787807814457e-17,
    "rel_l2": 8.142731530824676e-06
  },
  "Y": {
    "max_abs": 6.938893903907228e-18,
    "rel_l2": 1.2130105915275635e-06
  },
  "Z": {
    "max_abs": 1.3877787807814457e-17,
    "rel_l2": 1.1136081639051554e-06
  }
}


## 5. Record or compare a local version baseline

The stored fixture contains:

- six production one-arm arrays;
- production \(X,Y,Z\);
- source-control and environment metadata.

The files are saved beside the notebook with a `_local_wdm_` prefix. Add the
single `_local_wdm_*` rule to `.gitignore` so these machine-specific outputs
cannot be committed accidentally.

In [7]:
baseline_arrays = (
    NOTEBOOK_DIR / f"_local_wdm_01_{BASELINE_NAME}_outputs.npz"
)
baseline_metadata = (
    NOTEBOOK_DIR / f"_local_wdm_01_{BASELINE_NAME}_metadata.json"
)
validation_report = NOTEBOOK_DIR / "_local_wdm_01_validation_report.json"

ordered_links = tuple(production_wave["link_w"])
current_arrays = {
    **{
        f"link_{sender}{receiver}": np.asarray(
            production_wave["link_w"][(sender, receiver)]
        )
        for sender, receiver in ordered_links
    },
    "X": np.asarray(production_xyz[0]),
    "Y": np.asarray(production_xyz[1]),
    "Z": np.asarray(production_xyz[2]),
}

source_state = git_state(WDM_LISAresponse.__file__)
version_comparison = None

if BASELINE_ACTION == "record":
    if baseline_arrays.exists() and not ALLOW_BASELINE_OVERWRITE:
        raise FileExistsError(
            f"Baseline already exists: {baseline_arrays}. "
            "Use BASELINE_ACTION='compare', choose another name, or "
            "set ALLOW_BASELINE_OVERWRITE=True."
        )

    np.savez_compressed(baseline_arrays, **current_arrays)
    baseline_metadata.write_text(
        json.dumps(
            json_ready(
                {
                    "source": source_state,
                    "configuration": {
                        "N": N,
                        "dt": dt,
                        "Nf": Nf,
                        "L_trunc": L_TRUNC,
                        "calc_m0": CALC_M0,
                        "production_shift": PRODUCTION_SHIFT,
                    },
                    "ordered_links": ordered_links,
                    "orbit_file": str(ORBIT_FILE),
                }
            ),
            indent=2,
        ),
        encoding="utf-8",
    )
    print("Recorded baseline:", baseline_arrays)
    print("Recorded metadata:", baseline_metadata)

elif BASELINE_ACTION == "compare":
    if not baseline_arrays.exists():
        raise FileNotFoundError(
            f"No baseline found at {baseline_arrays}. "
            "Run once with BASELINE_ACTION='record'."
        )

    version_comparison = {}
    with np.load(baseline_arrays, allow_pickle=False) as baseline:
        for name, current in current_arrays.items():
            version_comparison[name] = compare_arrays(
                current,
                baseline[name],
                m_start=M_START,
            )

    print("Current revision versus saved baseline:")
    print(json.dumps(json_ready(version_comparison), indent=2))

elif BASELINE_ACTION != "none":
    raise ValueError(
        "BASELINE_ACTION must be 'record', 'compare', or 'none'."
    )

report = {
    "source": source_state,
    "orbit_file": str(ORBIT_FILE),
    "production_reference_links": production_reference_links,
    "production_reference_xyz": production_reference_xyz,
    "scheduling_equivalence": scheduling_comparison,
    "prepared_batch_map": prepared_batch_map,
    "version_baseline_action": BASELINE_ACTION,
    "version_comparison": version_comparison,
}

validation_report.write_text(
    json.dumps(json_ready(report), indent=2),
    encoding="utf-8",
)
print("Saved validation report:", validation_report)

Recorded baseline: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples/_local_wdm_01_clean_one_arm_v1_outputs.npz
Recorded metadata: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples/_local_wdm_01_clean_one_arm_v1_metadata.json
Saved validation report: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples/_local_wdm_01_validation_report.json
